# CineSage
- A hybrid movie recommendation system based on Neural Collabrative Filtering using Generalized Matrix Function and Multilayered Perceptron along with content based filtering using cosine similarities to handle cold start problem

### Pre-requisited

In [1]:
import warnings
warnings.filterwarnings(action='ignore')

### Import necessary libraries

In [2]:
import os, json, joblib, random, time
from pathlib import Path
import numpy as np
import pandas as pd

In [3]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from tqdm import tqdm

In [4]:
from sentence_transformers import SentenceTransformer
import scipy.sparse as sp

### Import custom modules

In [5]:
import sys
sys.path.append(str(Path.cwd().parent / 'src'))

In [6]:
import sys
sys.path.append(str(Path.cwd().parent / 'src'))
from data import load_merged_df, build_mappings
from content import build_sentence_embeddings, get_topk_content
from model import NCF
from train import train_ncf_bpr
from eval import hit_ratio, ndcg, evaluate_model
from utils import save_artifact, load_artifact

### File Paths

In [7]:
MOVIE_FILE = r"G:\CineSage\data\raw\ml-latest-small\ratings.csv"
LANGUAGE_FILE = r'G:\CineSage\data\raw\tmdb_languages.json'
RATINGS_FILE = r'G:\CineSage\data\raw\ml-latest-small\ratings.csv'
MERGED_FILE = r'G:\CineSage\data\processed\merged_movielens_tmdb.csv'
OUT_DIR = '../saved_models'
os.makedirs(OUT_DIR, exist_ok=True)

### 1) Load merged data

In [8]:
df = load_merged_df(merged_path=MERGED_FILE,
                    ratings_path=RATINGS_FILE,
                    movies_json_path=MOVIE_FILE)
print("Loaded", df.shape)
df.head()

Loaded (34066, 9)


,user_id,movie_id,rating,timestamp,title,release_date,genre_names,original_language_full,overview
0,1,3,4.0,964981247,Shadows in Paradise,1986-10-17,"['Comedy', 'Drama', 'Romance']",Finnish,"Nikander, a rubbish collector and would-be ent..."
1,1,6,4.0,964982224,Judgment Night,1993-10-15,"['Action', 'Crime', 'Thriller']",English,"Four young friends, while taking a shortcut en..."
2,1,70,3.0,964982400,Million Dollar Baby,2004-12-05,['Drama'],English,Despondent over a painful estrangement from hi...
3,1,101,5.0,964980868,Léon: The Professional,1994-09-14,"['Crime', 'Drama', 'Action']",French,"Léon, the top hit man in New York, has earned ..."
4,1,110,4.0,964982176,Three Colors: Red,1994-05-12,"['Drama', 'Mystery', 'Romance']",French,Part-time model Valentine unexpectedly befrien...


#### Quick EDA

In [9]:
print("Unique users:", df['user_id'].nunique())
print("Unique movies:", df['movie_id'].nunique())
print("Ratings range:", df['rating'].min(), df['rating'].max())
print("Sample genres:", df['genre_names'].dropna().sample(5).tolist())
print("Duplicates:", df.duplicated().sum())
print("Missing values:", df.isna().sum().sum())

Unique users: 610
Unique movies: 2497
Ratings range: 0.5 5.0
Sample genres: ["['Crime', 'Thriller', 'Action']", "['Drama']", "['Drama', 'Mystery']", "['Drama', 'Thriller', 'War']", "['Drama', 'Romance']"]
Duplicates: 0
Missing values: 124


### Preprocessing

In [10]:
# Build user/item mappings
user2idx, idx2user, movie2idx, idx2movie = build_mappings(df)
num_users = len(user2idx)
num_items = len(movie2idx)
print("num_users:", num_users, "num_items:", num_items)

# Map to indices for fast processing
df['user_idx'] = df['user_id'].map(user2idx)
df['movie_idx'] = df['movie_id'].map(movie2idx)

num_users: 610 num_items: 2497


### 3) Sentence-transformer pipeline for cold-start/cold-item

In [11]:
# Prepare content strings for TF-IDF (genres + overview)
def preprocess_content(row):
    genres = row['genre_names']
    if isinstance(genres, str) and genres.startswith('['):
        genres = " ".join(eval(genres))
    else:
        genres = str(genres)
    overview = row['overview'] if isinstance(row['overview'], str) else ""
    return f"{genres} {overview}".strip()

df['content'] = df.apply(preprocess_content, axis=1)
df_unique = df.drop_duplicates(subset=['movie_id']).reset_index(drop=True)

content_embeddings = build_sentence_embeddings(
    df_unique, field='content', model_name='all-MiniLM-L6-v2'
)
joblib.dump(content_embeddings, os.path.join(OUT_DIR, 'content_embeddings.joblib'))
print("Content embeddings shape:", content_embeddings.shape)

Batches: 100%|█████████████████████████████████████████████████████████████████████████| 79/79 [00:33<00:00,  2.36it/s]

Content embeddings shape: (2497, 384)


### 4) Holdout (Train-Test Split) For NCF  

In [12]:
def leave_one_out(df):
    """
    Leave-one-out split by timestamp: last item for each user goes to test set.
    """
    df_sorted = df.sort_values(['user_idx', 'timestamp'])
    train_idx, test_idx = [], []
    for uid, group in df_sorted.groupby('user_idx'):
        if len(group) < 2:
            train_idx.extend(group.index.tolist())
            continue
        test_idx.append(group.tail(1).index.item())
        train_idx.extend(group.head(len(group) - 1).index.tolist())
    return df.loc[train_idx], df.loc[test_idx]

In [13]:
train_df, test_df = leave_one_out(df)
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

Train size: 33456, Test size: 610


### 5) Dataset Class

In [14]:
class BPRDataset(Dataset):
    """
    Bayesian Personalized Ranking Dataset.
    Samples (user, positive_item, negative_item) triplets for training.
    """
    def __init__(self, df, num_items):
        # Map user → set of positive items
        self.user_item_dict = df.groupby('user_idx')['movie_idx'].apply(set).to_dict()
        self.users = df['user_idx'].values
        self.items = df['movie_idx'].values
        self.num_items = num_items

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        user = self.users[idx]
        pos_item = self.items[idx]
        # Sample until we get a true negative
        while True:
            neg_item = np.random.randint(0, self.num_items)
            if neg_item not in self.user_item_dict.get(user, set()):
                break
        return torch.tensor(user), torch.tensor(pos_item), torch.tensor(neg_item)

In [15]:
train_ds = BPRDataset(train_df, num_items)

### 6) Build NCF Model

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = NCF(num_users=num_users, num_items=num_items, embedding_dim=32)
model.to(device)

NCF(
  (user_embed_gmf): Embedding(610, 32)
  (item_embed_gmf): Embedding(2497, 32)
  (user_embed_mlp): Embedding(610, 32)
  (item_embed_mlp): Embedding(2497, 32)
  (mlp): Sequential(
    (0): Linear(in_features=64, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
  )
  (output_layer): Linear(in_features=64, out_features=1, bias=True)
)

### 7) Model Trainning

In [17]:
model = train_ncf_bpr(model, train_ds, device=device,
                      epochs=20, batch_size=256, lr=0.01, weight_decay = 1e-4,
                      save_path=os.path.join(OUT_DIR, 'ncf_bpr_checkpoint.pt'))


Epoch 1/20, BPR Loss: 0.5148
Epoch 2/20, BPR Loss: 0.3404
Epoch 3/20, BPR Loss: 0.3266
Epoch 4/20, BPR Loss: 0.3226
Epoch 5/20, BPR Loss: 0.3130
Epoch 6/20, BPR Loss: 0.3109
Epoch 7/20, BPR Loss: 0.3053
Epoch 8/20, BPR Loss: 0.2975
Epoch 9/20, BPR Loss: 0.2842
Epoch 10/20, BPR Loss: 0.2698
Epoch 11/20, BPR Loss: 0.2578
Epoch 12/20, BPR Loss: 0.2543
Epoch 13/20, BPR Loss: 0.2451
Epoch 14/20, BPR Loss: 0.2376
Epoch 15/20, BPR Loss: 0.2348
Epoch 16/20, BPR Loss: 0.2266
Epoch 17/20, BPR Loss: 0.2256
Epoch 18/20, BPR Loss: 0.2186
Epoch 19/20, BPR Loss: 0.2162
Epoch 20/20, BPR Loss: 0.2184


### 8) Evaluation

In [18]:
from eval import evaluate_model

K = 10
hr, ndcg, precision, recall = evaluate_model(
    model, 
    train_df.rename(columns={'user_idx': 'user', 'movie_idx': 'movie'}), 
    test_df.rename(columns={'user_idx': 'user', 'movie_idx': 'movie'}), 
    num_items=num_items, 
    K=K, 
    device=device
)

Evaluating: 100%|███████████████████████████████████████████████████████████████████| 610/610 [00:00<00:00, 615.16it/s]


In [25]:
print(f"Hit Ratio@{K}: {100 * hr:.2f}%")
print(f"NDCG@{K}: {100* ndcg:.2f}%")
print(f"Precision@{K}: {100* precision:.2f}%")
print(f"Recall@{K}: {100 *recall:.2f}%")


Hit Ratio@10: 60.98%
NDCG@10: 35.83%
Precision@10: 6.10%
Recall@10: 60.98%


### 9) Content-based recommendation function for cold-start/new users

In [20]:
def recommend_for_new_user(last_movie_id, df, embeddings, top_k=10):
    if last_movie_id not in movie2idx:
        return []

    idx_list = df.index[df['movie_id'] == last_movie_id].tolist()
    if not idx_list:
        return []
    idx = idx_list[0]

    top_indices = get_topk_content(idx, embeddings, top_k + 1)

    seen_ids = {last_movie_id}
    res = []
    for i in top_indices:
        movie_id = int(df.iloc[i]['movie_id'])
        if movie_id in seen_ids:
            continue
        seen_ids.add(movie_id)
        res.append({
            'movie_id': movie_id,
            'title': df.iloc[i]['title']
        })
        if len(res) >= top_k:
            break
    return res

In [21]:
content_embeddings = joblib.load(os.path.join(OUT_DIR, 'content_embeddings.joblib'))

sample_last = 1573
print("Last watched sample:", sample_last)
print(recommend_for_new_user(sample_last, df_unique, content_embeddings, top_k=5))

Last watched sample: 1573
[{'movie_id': 562, 'title': 'Die Hard'}, {'movie_id': 1571, 'title': 'Live Free or Die Hard'}, {'movie_id': 2320, 'title': 'Executive Decision'}, {'movie_id': 1572, 'title': 'Die Hard: With a Vengeance'}, {'movie_id': 2034, 'title': 'Training Day'}]
